# DAGs vs. Agents: Architecting Reliability
Should you use a deterministic Workflow (DAG) or a non-deterministic Agent? SOTA systems use **Agentic Workflows** to combine both.

## 1. Deterministic DAG (Airflow/Scripts)
Strict, reliable, but brittle. It cannot handle edge cases.

In [ ]:
def refund_workflow(user_id: int):
    print(f"Step 1: Lookup User {user_id}")
    print(f"Step 2: Reverse transaction in Stripe")
    print(f"Step 3: Send generic confirmation email")
    return "Complete"

print("Standard Execution:")
refund_workflow(99)

Standard Execution:
Step 1: Lookup User 99
Step 2: Reverse transaction in Stripe
Step 3: Send generic confirmation email


## 2. Agentic Workflow (The SOTA Pattern)
We use a deterministic graph to enforce the safety of the `refund_workflow`, but we use an Agent as the *Router* at the very beginning to handle the messy human language.

In [ ]:
def llm_intent_router(email_text: str) -> str:
    print(f"🤖 LLM reading messy email: '{email_text}'")
    if "refund" in email_text.lower() or "money back" in email_text.lower():
        return "REFUND_WORKFLOW"
    return "SUPPORT_WORKFLOW"

def agentic_workflow_execution(email: str, user_id: int):
    # The Agent handles the unpredictability of human text
    route = llm_intent_router(email)
    
    # The DAG enforces strict execution
    if route == "REFUND_WORKFLOW":
        print(f"🔀 Routing to deterministic Refund DAG...")
        refund_workflow(user_id)
    else:
        print(f"🔀 Routing to human support queue.")

print("--- Edge Case 1 ---")
agentic_workflow_execution("I hate this product, give me my money back immediately!", 102)

print("\n--- Edge Case 2 ---")
agentic_workflow_execution("How do I change my password?", 102)

--- Edge Case 1 ---
🤖 LLM reading messy email: 'I hate this product, give me my money back immediately!'
🔀 Routing to deterministic Refund DAG...
Step 1: Lookup User 102
Step 2: Reverse transaction in Stripe
Step 3: Send generic confirmation email

--- Edge Case 2 ---
🤖 LLM reading messy email: 'How do I change my password?'
🔀 Routing to human support queue.
